In [1]:
import os

In [2]:
%pwd

'e:\\Python Projects\\Deep_learning\\mlops_test1\\research'

In [3]:
os.chdir('../')

Lets create the configuration entity for model evaluation config

In [4]:
from dataclasses import dataclass
from pathlib import Path
from box import ConfigBox
@dataclass(frozen= True)
class ModelEvaluationConfig():
    path_of_model:Path
    training_data: Path
    all_params: ConfigBox
    params_image_size: list
    params_batch_size: int
    metrics_file_path: Path




Create the configuration Manager

In [5]:
from src.chicken_disease_classification.constants import CONFIG_FILE_PATH,PARAMS_FILE_PATH
from src.chicken_disease_classification.utils.common import read_yaml, create_directories, save_json
import os

class ConfigurationManager():

    def __init__(self,config_file_path
                 , params_file_path):
        self.config= read_yaml(config_file_path)
        self.params= read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self)-> ModelEvaluationConfig :
        config= self.config
        params= self.params
        training_data= os.path.join(self.config.data_ingestion.unzip_dir,
                                    os.path.basename(self.config.data_ingestion.source_URL))
        create_directories([Path(config.evaluation.root_directory)])
        
        return ModelEvaluationConfig(path_of_model= Path(config.training.model_file_path)
                                     ,training_data= Path(training_data)
                                     ,all_params= params 
                                     ,params_image_size=params.IMAGE_SIZE
                                     ,params_batch_size=params.BATCH_SIZE
                                     ,metrics_file_path= Path(config.evaluation.metrics_file_path)                                     
                                     )
    



    

Create evaluation component

In [6]:
import torch
import torch.nn as nn
from torchvision.transforms import transforms
import numpy as np
from zipfile import ZipFile
import urllib.request as request
import os
import time
from pathlib import Path
from src.chicken_disease_classification.entity.config_entity import TrainingConfig
import torch.optim.lr_scheduler as lr_scheduler
from torchvision.datasets import ImageFolder
from torch.utils.data.dataloader import DataLoader
import copy
from src.chicken_disease_classification import logger
from src.chicken_disease_classification.utils.common import save_json

In [7]:
class Evaluation:
    def __init__(self,evaluation_config: ModelEvaluationConfig):
        self.config= evaluation_config

    def load_model(self,model_path):
        self.model= torch.load(model_path,weights_only= False)

    def preprocess_data(self):
        mean= np.array([0.5,0.5,0.5])
        std= np.array([0.25,0.25,0.25])

        self.data_transforms= {
            'train': transforms.Compose([ transforms.RandomResizedCrop(224),
                                        transforms.RandomHorizontalFlip(1),
                                        transforms.ToTensor(),
                                        transforms.Normalize(mean,std)]),
            'val':transforms.Compose([transforms.Resize(256),
                                    transforms.CenterCrop(224),
                                    transforms.ToTensor(),
                                    transforms.Normalize(mean= mean,std= std)])
        }

        data_root= self.config.training_data

        self.testing_dataset=ImageFolder(root= os.path.join(data_root,'val'),\
                                            transform= self.data_transforms['val'])
    
        self.test_image_dataloader= torch.utils.data.DataLoader(dataset= self.testing_dataset,\
                                                        batch_size= self.config.params_batch_size, 
                                                        shuffle= False)

        self.dataset_size = len(self.testing_dataset) 
        self.class_names = self.testing_dataset.classes

    def build_additional_parameters(self):
       self.criterion= nn.CrossEntropyLoss()

    @staticmethod
    def evaluation(model,criterion,dataloader,dataset_size):
        # Get model filepath
        running_loss= 0.0
        running_corrects= 0.0
        
        for inputs,labels in dataloader:
            with torch.no_grad():
                outputs= model(inputs)
                _,predictions=torch.max(outputs,1)
                loss= criterion(outputs,labels)

            #statistics:
            running_loss+= loss.item()*inputs.size(0)
            running_corrects+= torch.sum(predictions==labels.data)
        
        epoch_loss= running_loss/dataset_size
        epoch_accuracy= running_corrects/dataset_size

        logger.info(msg='Loss: {:.4f} Acc: {:.4f}'.format(epoch_loss, epoch_accuracy))

        metrics_dict= dict(loss= epoch_loss,accuracy=epoch_accuracy.item())

        return metrics_dict
    
    @staticmethod
    def save_metrics(metrics_dict,path):
        save_json(path=path,data= metrics_dict)






                







Get the evaluation pipeline

In [8]:
class ModelEvaluationPipeline():
    def __init__(self):
        pass
    def main(self):
        try:
            configuration_manager= ConfigurationManager(CONFIG_FILE_PATH,PARAMS_FILE_PATH)
            model_eval_config= configuration_manager.get_model_evaluation_config()
            model_evaluator= Evaluation(evaluation_config= model_eval_config)
            model_evaluator.load_model(model_evaluator.config.path_of_model)
            model_evaluator.preprocess_data()
            model_evaluator.build_additional_parameters()
            metrics_dict= Evaluation.evaluation(model= model_evaluator.model
                                    ,criterion= model_evaluator.criterion
                                    ,dataloader=model_evaluator.test_image_dataloader
                                    ,dataset_size=model_evaluator.dataset_size)
            print(metrics_dict)
            print(type(metrics_dict))
            Evaluation.save_metrics(metrics_dict=metrics_dict,path=model_eval_config.metrics_file_path)
        except Exception as e:
            logger.error(e)
            raise e



In [9]:
STAGE_NAME= "Evaluation Pipeline"
try:
    logger.info(f">>>>>>>>>>>>>>>>>>>>>STAGE {STAGE_NAME} Started <<<<<<<<<<<<<<<<<<<<<<<<")
    eval_pipeline=ModelEvaluationPipeline()
    eval_pipeline.main()
    logger.info(f">>>>>>>>>>>>>>>>>>>>>STAGE {STAGE_NAME} Completed <<<<<<<<<<<<<<<<<<<<<<<<")
except Exception as e:
    logger.error(f'STAGE: {STAGE_NAME} failed with the following exception:{e}')
    raise e


2025-05-28 22:45:41,383 - INFO - >>>>>>>>>>>>>>>>>>>>>STAGE Evaluation Pipeline Started <<<<<<<<<<<<<<<<<<<<<<<<
2025-05-28 22:45:41,385 - INFO - yaml file: e:\Python Projects\Deep_learning\mlops_test1\config\config.yaml loaded successfully
2025-05-28 22:45:41,385 - INFO - yaml file: e:\Python Projects\Deep_learning\mlops_test1\params.yaml loaded successfully
2025-05-28 22:45:41,385 - INFO - created directory at: artifacts
2025-05-28 22:45:41,385 - INFO - created directory at: artifacts\evaluation
2025-05-28 22:45:43,770 - INFO - Loss: 0.1366 Acc: 0.9739
{'loss': 0.13661579359201045, 'accuracy': 0.9738562107086182}
<class 'dict'>
2025-05-28 22:45:43,770 - INFO - json file saved at: artifacts\evaluation\scores.json
2025-05-28 22:45:43,770 - INFO - >>>>>>>>>>>>>>>>>>>>>STAGE Evaluation Pipeline Completed <<<<<<<<<<<<<<<<<<<<<<<<
